# When sessions don't add up

Sessions in Google's Merchandise Store sample, November 2020 to January 2021, counted three ways
for the explorable: as one total, by device category (the rows add up to the total) and by page
title (the rows add up to more than the total).

Run all cells from the top. The CSVs are saved to `data/raw/sessions-dont-add-up/`.
Setup is in `data/README.md`.

In [1]:
import os
from pathlib import Path
import pandas as pd

from dotenv import find_dotenv, load_dotenv
from google.cloud import bigquery

load_dotenv(find_dotenv(usecwd=True))
project_id = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not project_id:
    raise RuntimeError("Set GOOGLE_CLOUD_PROJECT in a .env file (see data/.env.example).")

# Queries run in your own project: billed there, or counted against the free sandbox quota.
client = bigquery.Client(project=project_id)

# Paths are relative to this notebook's folder.
EXPLORABLE = "sessions-dont-add-up"
SQL_DIR = Path("../sql") / EXPLORABLE
RAW_DIR = Path("../raw") / EXPLORABLE


def run_query(name):
    """Run sql/<explorable>/<name>.sql and return the result as a DataFrame."""
    sql = (SQL_DIR / f"{name}.sql").read_text(encoding="utf-8")
    # A dry run is free and shows the cost before the real run.
    estimate = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True))
    print(f"{name}: {estimate.total_bytes_processed / 1024**3:.2f} GB to read")
    return client.query(sql).to_dataframe()


def save(df, name):
    # LF line endings on every OS, so a re-run matches the committed file byte for byte.
    df.to_csv(RAW_DIR / f"{name}.csv", index=False, encoding="utf-8", lineterminator="\n")

In [2]:
real_sessions = run_query("real_sessions")
by_device = run_query("sessions_by_device")
by_page = run_query("sessions_by_page")

real_sessions: 0.93 GB to read
sessions_by_device: 0.96 GB to read
sessions_by_page: 1.45 GB to read


In [3]:
# The numbers shown on the site. The sample never changes, so a re-run must match them.
real_total = real_sessions.sessions.iloc[0]
device_total = by_device.sessions.sum()
page_total = by_page.sessions.sum()

assert real_total == 333_683
assert page_total == 884_352

# Device is session-scoped, so its rows must add up exactly; page is event-scoped, so they can't.
assert device_total == real_total

summary = pd.DataFrame(
    {"sessions": [real_total, device_total, page_total]},
    index=["Real sessions", "Sum of device rows", "Sum of page rows"],
)
summary["vs_real_total"] = summary["sessions"] - real_total
summary

,sessions,vs_real_total
Real sessions,333683,0
Sum of device rows,333683,0
Sum of page rows,884352,550669


In [4]:
save(real_sessions, "real_sessions")
save(by_device, "sessions_by_device")
save(by_page, "sessions_by_page")

If the data is unchanged, `git status data/raw` shows no changes: the new CSVs match the committed ones.
